# Dave's Trading System: Bounce 5, Predator Play, Fibonacci & Trinity

**Instruments:** SPY (S&P 500 ETF), QQQ (Nasdaq 100 ETF) — Alpaca paper trading

**Timeframe:** 1-min or 5-min bars

**Moving Averages:** 50-period MA (short-term) | 200-period MA (long-term)

---

## Strategies

| # | Strategy | Season | Win Rate |
|---|----------|--------|----------|
| 1 | Season Detector | All | Foundation |
| 2 | Bounce 5 Channel Play | Season 1 Ranging | 78% |
| 3 | Predator Play | Season 2 Trending | 90% |
| 4 | Fibonacci Retracement | All | High |
| 5 | Trinity Play | All | A+ Setup |

> Paper trading only. Always test before going live.

## Step 1: Enter Your Alpaca API Keys
Get your keys from https://app.alpaca.markets → API Keys

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────
API_KEY    = "YOUR_API_KEY_HERE"
API_SECRET = "YOUR_SECRET_KEY_HERE"

PAPER    = True   # Always True for paper trading
LIVE_MODE = True  # True = bot will place real paper orders

SYMBOLS  = ["SPY", "QQQ"]  # Scan both S&P500 and Nasdaq ETFs
TIMEFRAME = "1Min"         # Bar timeframe
# ────────────────────────────────────────────────────────────────

## Step 2: Install & Import Libraries

In [ ]:
!pip install alpaca-py --quiet

In [ ]:
import time
import datetime
import numpy as np
import pandas as pd

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

print("Libraries loaded successfully")

## Step 3: Connect to Alpaca

In [ ]:
trade_client = TradingClient(API_KEY, API_SECRET, paper=PAPER)
data_client  = StockHistoricalDataClient(API_KEY, API_SECRET)

account = trade_client.get_account()
print(f"Connected! Account: {account.account_number}")
print(f"Buying Power: ${float(account.buying_power):,.2f}")
print(f"Portfolio Value: ${float(account.portfolio_value):,.2f}")
print(f"Paper Trading: {PAPER}")
print(f"Live Mode: {LIVE_MODE}")

## Step 4: Season Detector
- **Season 1 RANGING**: 50 MA and 200 MA flat and intertwined → USE Bounce 5
- **Season 2 TRENDING**: 50 MA clearly separated from 200 MA → USE Predator Play
- **Season 3 WHIPSAW**: Price violently crossing both MAs → STAND ASIDE

In [ ]:
def get_bars(symbol, lookback_bars=250):
    end   = datetime.datetime.now(datetime.timezone.utc)
    start = end - datetime.timedelta(minutes=lookback_bars * 2)
    req = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=TimeFrame.Minute,
        start=start,
        end=end,
        limit=lookback_bars
    )
    bars = data_client.get_stock_bars(req)
    df = bars.df.reset_index()
    if 'symbol' in df.columns:
        df = df[df['symbol'] == symbol].copy()
    df = df.sort_values('timestamp').tail(lookback_bars)
    df['close'] = df['close'].astype(float)
    df['open']  = df['open'].astype(float)
    df['high']  = df['high'].astype(float)
    df['low']   = df['low'].astype(float)
    return df

def detect_season(df, slope_thresh=0.0002, sep_thresh=0.003):
    closes = df['close'].values
    if len(closes) < 200:
        return 'UNKNOWN', 0.0, 0.0
    ma50  = np.mean(closes[-50:])
    ma200 = np.mean(closes[-200:])
    # Slope of 50 MA over last 10 bars
    ma50_prev  = np.mean(closes[-60:-10])
    ma50_slope = (ma50 - ma50_prev) / ma50_prev if ma50_prev != 0 else 0
    # Separation between MAs
    separation = abs(ma50 - ma200) / ma200 if ma200 != 0 else 0
    # Whipsaw: price crossing MAs rapidly
    recent = closes[-20:]
    crossings = sum(1 for i in range(1, len(recent))
                    if (recent[i-1] < ma50) != (recent[i] < ma50))
    if crossings >= 4:
        return 'WHIPSAW', ma50, ma200
    if separation < sep_thresh and abs(ma50_slope) < slope_thresh:
        return 'RANGING', ma50, ma200
    return 'TRENDING', ma50, ma200

print("Season detector ready")

## Step 5: Bounce 5 Channel Play (Season 1 — 78% Win Rate)
SHORT at Bounce 5 resistance. Target = Bounce 4 support. Stop = higher high above channel.

In [ ]:
def find_bounce5_setup(df):
    """Detect a Bounce 5 short setup. Returns dict or None."""
    closes = df['close'].values
    highs  = df['high'].values
    lows   = df['low'].values

    if len(closes) < 60:
        return None

    recent = closes[-60:]
    r_highs = highs[-60:]
    r_lows  = lows[-60:]

    # Find local swing highs (resistance bounces)
    swing_highs = []
    for i in range(2, len(recent) - 2):
        if (r_highs[i] > r_highs[i-1] and r_highs[i] > r_highs[i-2] and
            r_highs[i] > r_highs[i+1] and r_highs[i] > r_highs[i+2]):
            swing_highs.append((i, r_highs[i]))

    # Find local swing lows (support bounces)
    swing_lows = []
    for i in range(2, len(recent) - 2):
        if (r_lows[i] < r_lows[i-1] and r_lows[i] < r_lows[i-2] and
            r_lows[i] < r_lows[i+1] and r_lows[i] < r_lows[i+2]):
            swing_lows.append((i, r_lows[i]))

    if len(swing_highs) < 2 or len(swing_lows) < 2:
        return None

    # Use last 2 swing highs as resistance, last 2 swing lows as support
    resistance = np.mean([h[1] for h in swing_highs[-2:]])
    support    = np.mean([l[1] for l in swing_lows[-2:]])
    current    = closes[-1]

    channel_height = resistance - support
    if channel_height <= 0:
        return None

    # Channel angle check (must be <= 45 degrees, i.e. roughly flat)
    x_span = swing_highs[-1][0] - swing_highs[0][0]
    y_span = abs(swing_highs[-1][1] - swing_highs[0][1])
    angle  = np.degrees(np.arctan2(y_span, x_span)) if x_span > 0 else 90
    if angle > 45:
        return None

    # Price must be near resistance (within 0.3% of top of channel)
    proximity = (resistance - current) / channel_height
    if proximity > 0.30:
        return None

    # R:R check — minimum 4:1
    entry  = current
    target = support
    stop   = resistance * 1.002  # 0.2% above resistance
    risk   = stop - entry
    reward = entry - target
    if risk <= 0 or (reward / risk) < 4.0:
        return None

    return {
        'setup':      'BOUNCE5_SHORT',
        'entry':      round(entry, 4),
        'target':     round(target, 4),
        'stop':       round(stop, 4),
        'resistance': round(resistance, 4),
        'support':    round(support, 4),
        'rr_ratio':   round(reward / risk, 1),
        'channel_h':  round(channel_height, 4)
    }

print("Bounce 5 detector ready")

## Step 6: Predator Play (Season 2 — 90% Win Rate)
Price crosses BELOW 200 MA by distance D. LONG at 2×D below MA. Target = back above 200 MA.

In [ ]:
def find_predator_setup(df, ma200):
    """Detect a Predator Play long setup."""
    closes = df['close'].values
    current = closes[-1]

    if current >= ma200:
        return None  # Price must be BELOW 200 MA

    distance_D = ma200 - current
    entry_zone = ma200 - (2 * distance_D)

    # Price must be near the 2D entry zone (within 0.5%)
    if abs(current - entry_zone) / entry_zone > 0.005:
        return None

    target = ma200 * 1.001  # Slightly above 200 MA
    stop   = entry_zone * (1 - 0.005)
    risk   = current - stop
    reward = target - current

    if risk <= 0 or (reward / risk) < 2.0:
        return None

    return {
        'setup':       'PREDATOR_LONG',
        'entry':       round(current, 4),
        'target':      round(target, 4),
        'stop':        round(stop, 4),
        'distance_D':  round(distance_D, 4),
        'entry_zone':  round(entry_zone, 4),
        'ma200':       round(ma200, 4),
        'rr_ratio':    round(reward / risk, 1)
    }

print("Predator Play detector ready")

## Step 7: Fibonacci Retracement
Draw A→B move. 61.8% = highest probability (wholesale). 50% = moderate. 38.2% = lower.

In [ ]:
def find_fib_levels(df):
    """Calculate Fibonacci retracement levels from recent swing."""
    highs  = df['high'].values[-60:]
    lows   = df['low'].values[-60:]
    closes = df['close'].values

    swing_high = np.max(highs)
    swing_low  = np.min(lows)
    diff       = swing_high - swing_low

    if diff <= 0:
        return None

    levels = {
        'swing_high': round(swing_high, 4),
        'swing_low':  round(swing_low, 4),
        '38.2':       round(swing_high - 0.382 * diff, 4),
        '50.0':       round(swing_high - 0.500 * diff, 4),
        '61.8':       round(swing_high - 0.618 * diff, 4),
    }

    current = closes[-1]
    nearest = min(['38.2', '50.0', '61.8'], key=lambda k: abs(levels[k] - current))
    dist    = abs(levels[nearest] - current) / current

    levels['current']        = round(current, 4)
    levels['nearest_fib']    = nearest
    levels['distance_pct']   = round(dist * 100, 3)
    levels['at_fib']         = dist < 0.003  # Within 0.3% of a fib level

    return levels

print("Fibonacci calculator ready")

## Step 8: Candlestick Confirmation
Require bearish reversal candle before shorting. Require bullish candle before going long.

In [ ]:
def check_candle_confirmation(df, direction='SHORT'):
    """Check for candlestick confirmation at entry zone."""
    if len(df) < 3:
        return False, 'NOT_ENOUGH_DATA'

    o1 = float(df['open'].iloc[-2]);  c1 = float(df['close'].iloc[-2])
    o2 = float(df['open'].iloc[-1]);  c2 = float(df['close'].iloc[-1])
    h2 = float(df['high'].iloc[-1]);  l2 = float(df['low'].iloc[-1])

    body2  = abs(c2 - o2)
    range2 = h2 - l2
    ratio  = body2 / range2 if range2 > 0 else 0

    if direction == 'SHORT':
        # Bearish engulfing
        if c1 > o1 and c2 < o2 and o2 >= c1 and c2 <= o1:
            return True, 'BEARISH_ENGULFING'
        # Shooting star
        upper_wick = h2 - max(o2, c2)
        if c2 < o2 and upper_wick > body2 * 2 and ratio < 0.4:
            return True, 'SHOOTING_STAR'
        # Simple bearish close
        if c2 < o2 and ratio > 0.5:
            return True, 'BEARISH_CLOSE'
    else:  # LONG
        # Bullish engulfing
        if c1 < o1 and c2 > o2 and o2 <= c1 and c2 >= o1:
            return True, 'BULLISH_ENGULFING'
        # Hammer
        lower_wick = min(o2, c2) - l2
        if c2 > o2 and lower_wick > body2 * 2 and ratio < 0.4:
            return True, 'HAMMER'
        # Simple bullish close
        if c2 > o2 and ratio > 0.5:
            return True, 'BULLISH_CLOSE'

    return False, 'NO_CONFIRMATION'

print("Candle confirmation ready")

## Step 9: Order Execution

In [ ]:
def place_order(symbol, side, qty, setup_info):
    """Place a market order if LIVE_MODE is True."""
    if not LIVE_MODE:
        print(f"  [SIM] Would {side} {qty} {symbol} | Setup: {setup_info['setup']}")
        return None

    try:
        order_side = OrderSide.SELL if side == 'SHORT' else OrderSide.BUY
        req = MarketOrderRequest(
            symbol=symbol,
            qty=qty,
            side=order_side,
            time_in_force=TimeInForce.DAY
        )
        order = trade_client.submit_order(req)
        print(f"  ✅ ORDER PLACED: {side} {qty} {symbol} @ market")
        print(f"     Order ID: {order.id}")
        print(f"     Target: ${setup_info['target']} | Stop: ${setup_info['stop']}")
        return order
    except Exception as e:
        print(f"  ❌ Order failed: {e}")
        return None

def calculate_qty(symbol, risk_dollars=50):
    """Calculate shares to trade based on risk amount."""
    try:
        account = trade_client.get_account()
        bp = float(account.buying_power)
        # Use at most 10% of buying power or $5000, whichever is less
        max_position = min(bp * 0.10, 5000)
        # Get current price
        req = StockBarsRequest(
            symbol_or_symbols=symbol,
            timeframe=TimeFrame.Minute,
            start=datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=5),
            end=datetime.datetime.now(datetime.timezone.utc),
            limit=1
        )
        bars = data_client.get_stock_bars(req)
        price = float(bars.df['close'].iloc[-1])
        qty = max(1, int(max_position / price))
        return qty
    except:
        return 1

print("Order execution ready")

## Step 10: Full Scanner — Runs Every 60 Seconds

In [ ]:
def scan_symbol(symbol, trade_count, daily_pnl):
    """Run full strategy scan for one symbol. Returns updated trade_count."""
    print(f"\n  [{symbol}] Fetching bars...")
    try:
        df = get_bars(symbol, lookback_bars=250)
    except Exception as e:
        print(f"  [{symbol}] Error fetching data: {e}")
        return trade_count

    if len(df) < 200:
        print(f"  [{symbol}] Not enough bars ({len(df)}), skipping")
        return trade_count

    # 1. Detect season
    season, ma50, ma200 = detect_season(df)
    current_price = float(df['close'].iloc[-1])
    print(f"  [{symbol}] Price=${current_price:.2f} | Season={season} | MA50=${ma50:.2f} | MA200=${ma200:.2f}")

    if season == 'WHIPSAW':
        print(f"  [{symbol}] ⚠️  WHIPSAW — standing aside")
        return trade_count

    # 2. Fibonacci levels
    fib = find_fib_levels(df)
    if fib:
        fib_tag = f"near {fib['nearest_fib']}%" if fib['at_fib'] else f"off fib ({fib['distance_pct']}% away)"
        print(f"  [{symbol}] Fib: {fib_tag}")

    # 3. Season 1 — Bounce 5 SHORT
    if season == 'RANGING':
        setup = find_bounce5_setup(df)
        if setup:
            confirmed, candle_type = check_candle_confirmation(df, 'SHORT')
            trinity = fib and fib['at_fib'] and fib['nearest_fib'] == '61.8'
            label = 'TRINITY A+' if trinity else 'Bounce 5'
            print(f"  [{symbol}] 🎯 {label} setup! R:R={setup['rr_ratio']}:1 | Candle={candle_type}")
            if confirmed:
                print(f"  [{symbol}] 🚀 ENTRY: SHORT @ ${setup['entry']} | Target=${setup['target']} | Stop=${setup['stop']}")
                qty = calculate_qty(symbol) * (2 if trinity else 1)
                order = place_order(symbol, 'SHORT', qty, setup)
                if order or not LIVE_MODE:
                    trade_count += 1
            else:
                print(f"  [{symbol}] ⏳ Waiting for candle confirmation...")
        else:
            print(f"  [{symbol}] No Bounce 5 setup yet")

    # 4. Season 2 — Predator Play LONG
    elif season == 'TRENDING':
        setup = find_predator_setup(df, ma200)
        if setup:
            confirmed, candle_type = check_candle_confirmation(df, 'LONG')
            print(f"  [{symbol}] 🦁 Predator Play! R:R={setup['rr_ratio']}:1 | Candle={candle_type}")
            if confirmed:
                print(f"  [{symbol}] 🚀 ENTRY: LONG @ ${setup['entry']} | Target=${setup['target']} | Stop=${setup['stop']}")
                qty = calculate_qty(symbol)
                order = place_order(symbol, 'LONG', qty, setup)
                if order or not LIVE_MODE:
                    trade_count += 1
            else:
                print(f"  [{symbol}] ⏳ Waiting for candle confirmation...")
        else:
            print(f"  [{symbol}] No Predator setup yet")

    return trade_count

print("Scanner ready")

## Step 11: Live Trading Loop
▶️ **Run this cell to start the bot.** It scans every 60 seconds during market hours (8:30 AM–3:00 PM Central Time).

In [ ]:
# ── LIVE TRADING LOOP ───────────────────────────────────────────
# Market hours: 8:30 AM to 3:00 PM Central Time (CT)
# CT = UTC-5 (winter/CST) or UTC-6 (summer/CDT)
# Minnesota is in Central Time zone
CT_OFFSET     = datetime.timedelta(hours=-5)  # CST (winter); change to -6 for CDT summer
CT_ZONE       = datetime.timezone(CT_OFFSET)

MARKET_OPEN   = datetime.time(8, 30)   # 8:30 AM CT
MARKET_CLOSE  = datetime.time(15, 0)   # 3:00 PM CT

DAILY_GOAL    = 100.0   # Stop trading after $100 profit
MAX_TRADES    = 3       # Max trades per day
SCAN_INTERVAL = 60      # Seconds between scans

print(f"\n{'='*55}")
print(f"  Dave's Bounce 5 Bot — STARTING")
print(f"  Live mode: {LIVE_MODE} | Paper: {PAPER}")
print(f"  Scanning: {SYMBOLS}")
print(f"  Market hours: {MARKET_OPEN} – {MARKET_CLOSE} CT")
print(f"{'='*55}")

trade_count = 0
daily_pnl   = 0.0

while True:
    now_ct   = datetime.datetime.now(CT_ZONE)
    now_time = now_ct.time()
    timestamp = now_ct.strftime('%I:%M:%S %p CT')

    # ── Market hours check ──────────────────────────────────────
    if not (MARKET_OPEN <= now_time <= MARKET_CLOSE):
        print(f"[{timestamp}] Market closed | Next open: {MARKET_OPEN} CT")
        time.sleep(SCAN_INTERVAL)
        continue

    # ── Daily limits check ──────────────────────────────────────
    if trade_count >= MAX_TRADES:
        print(f"[{timestamp}] Max trades ({MAX_TRADES}) reached today. Done.")
        break
    if daily_pnl >= DAILY_GOAL:
        print(f"[{timestamp}] Daily goal ${DAILY_GOAL} hit! PnL=${daily_pnl:.2f}. Done.")
        break

    print(f"\n[{timestamp}] Scanning... Trades={trade_count}/{MAX_TRADES}")

    # ── Scan each symbol ────────────────────────────────────────
    for symbol in SYMBOLS:
        try:
            trade_count = scan_symbol(symbol, trade_count, daily_pnl)
        except Exception as e:
            print(f"  [{symbol}] Scan error: {e}")

    print(f"\n  Sleeping {SCAN_INTERVAL}s...")
    time.sleep(SCAN_INTERVAL)
# ────────────────────────────────────────────────────────────────

## Notes & Next Steps

**Open items to refine (from Dave's content):**
- Exact stop loss point distances per strategy
- Full Trinity Play confluence rules
- Crack Run setup (referenced but not yet documented)

**Before going live:**
1. Run on paper trading for at least 2-4 weeks
2. Track every trade in a journal (entry, exit, R:R, season, pattern)
3. Verify the season detector matches what you see on your chart manually
4. Tune `slope_thresh` and `sep_thresh` in `detect_season()` to match your timeframe
5. Set `LIVE_MODE = True` only after you're confident in the signals